# THE WORST TRADE OF ALL TIME

Find the most lopsided trade based on **total career value given away**.

This measures what each side GAVE AWAY and calculates the full career points those assets scored from the trade date forward, regardless of who owned them after the trade.

## Methodology

This analysis uses the "hindsight is 20/20" approach:
- For **players**: Counts ALL career fantasy points scored from the trade date forward
- For **draft picks**: Counts ALL career points of the drafted player
- **Does NOT** filter by subsequent ownership - we measure the true long-term value lost

This answers: "What was the worst trade when evaluated by pure asset value given away?"

In [ ]:
spark.sql("USE CATALOG workspace")

## Top 5 Worst Trades

In [ ]:
top_worst_trades = spark.sql("""
SELECT
  transaction_id,
  trade_season,
  roster_a,
  roster_b,
  roster_a_manager,
  roster_b_manager,
  ROUND(roster_a_points, 1) as roster_a_points,
  ROUND(roster_b_points, 1) as roster_b_points,
  ROUND(point_differential, 1) as point_differential,
  winner_manager,
  loser_manager,
  ROUND(trade_impact_magnitude, 1) as trade_impact_magnitude,
  trade_is_complete
FROM workspace.sleeper_trades.agg_trade_winners_enriched
WHERE cluster_name = 'League of Inches'
  AND trade_is_complete = TRUE
ORDER BY trade_impact_magnitude DESC
LIMIT 5
""")

display(top_worst_trades)

## THE Worst Trade - Complete Breakdown

Shows what each side GAVE AWAY and the career points lost.

In [ ]:
# Get complete asset breakdown for the worst trade
worst_trade_breakdown = spark.sql(f"""
WITH worst_trade AS (
  SELECT 
    transaction_id, 
    trade_season,
    roster_a,
    roster_b,
    roster_a_manager,
    roster_b_manager
  FROM workspace.sleeper_trades.agg_trade_winners_enriched
  WHERE cluster_name = 'League of Inches'
    AND trade_is_complete = TRUE
  ORDER BY trade_impact_magnitude DESC
  LIMIT 1
),
-- Get RECEIVED players with their career points
received_players AS (
  SELECT
    wt.transaction_id,
    fa.side_roster_id,
    CASE 
      WHEN fa.side_roster_id = wt.roster_a THEN wt.roster_b_manager
      ELSE wt.roster_a_manager
    END as manager_gave_away,
    'player' as asset_type,
    p.full_name as asset_name,
    p.position,
    COALESCE(SUM(pp.points), 0) as total_career_points
  FROM worst_trade wt
  JOIN sleeper_trades.fact_trade_player_assets fa
    ON wt.transaction_id = fa.transaction_id
    AND fa.direction = 'incoming'
  JOIN sleeper_core.dim_players p ON fa.player_id = p.player_id
  LEFT JOIN sleeper_trades.fact_trade_player_points_multi_season pp
    ON fa.transaction_id = pp.transaction_id
    AND fa.side_roster_id = pp.side_roster_id
    AND fa.player_id = pp.player_id
  GROUP BY wt.transaction_id, fa.side_roster_id, wt.roster_a, wt.roster_b, 
           wt.roster_a_manager, wt.roster_b_manager, p.full_name, p.position
),
-- Get RECEIVED picks with career points - join to bridge and career points
received_picks AS (
  SELECT
    wt.transaction_id,
    fpa.side_roster_id,
    CASE
      WHEN fpa.side_roster_id = wt.roster_a THEN wt.roster_b_manager
      ELSE wt.roster_a_manager
    END as manager_gave_away,
    'pick' as asset_type,
    CASE
      WHEN bp.player_id IS NOT NULL THEN CONCAT(fpa.pick_season, ' Round ', fpa.round, ' (', p.full_name, ')')
      ELSE CONCAT(fpa.pick_season, ' Round ', fpa.round, ' (unrealized)')
    END as asset_name,
    p.position,
    COALESCE(SUM(pp.points), 0) as total_career_points
  FROM worst_trade wt
  JOIN sleeper_trades.fact_trade_pick_assets fpa
    ON wt.transaction_id = fpa.transaction_id
  LEFT JOIN sleeper_trades.bridge_trade_pick_to_player bp
    ON fpa.league_id = bp.league_id
    AND fpa.transaction_id = bp.transaction_id
    AND fpa.side_roster_id = bp.side_roster_id
    AND fpa.pick_season = bp.pick_season
    AND fpa.round = bp.round
  LEFT JOIN sleeper_trades.fact_trade_pick_points_career pp
    ON bp.transaction_id = pp.transaction_id
    AND bp.side_roster_id = pp.side_roster_id
    AND bp.player_id = pp.player_id
  LEFT JOIN sleeper_core.dim_players p ON bp.player_id = p.player_id
  GROUP BY wt.transaction_id, fpa.side_roster_id, wt.roster_a, wt.roster_b,
           wt.roster_a_manager, wt.roster_b_manager, fpa.pick_season, fpa.round,
           bp.player_id, p.full_name, p.position
)
SELECT
  manager_gave_away as manager_name,
  asset_type,
  asset_name,
  position,
  ROUND(total_career_points, 1) as career_points_given_away
FROM received_players
UNION ALL
SELECT
  manager_gave_away as manager_name,
  asset_type,
  asset_name,
  position,
  ROUND(total_career_points, 1) as career_points_given_away
FROM received_picks
ORDER BY manager_name, asset_type DESC, career_points_given_away DESC
""")

display(worst_trade_breakdown)

## Summary by Manager

In [ ]:
from pyspark.sql import functions as F

# Calculate total career points given away by each manager
manager_totals = worst_trade_breakdown.groupBy('manager_name').agg(
    F.sum('career_points_given_away').alias('total_career_points_given_away')
).orderBy(F.desc('total_career_points_given_away'))

display(manager_totals)

# Extract variables for visualizations
worst_trade_info = top_worst_trades.first()
loser = worst_trade_info['loser_manager']
winner = worst_trade_info['winner_manager']
worst_trade_season = worst_trade_info['trade_season']
magnitude = worst_trade_info['trade_impact_magnitude']
worst_trade_id = worst_trade_info['transaction_id']

## Visualization

In [ ]:
import matplotlib.pyplot as plt

# Convert to pandas for plotting
pdf = manager_totals.toPandas()

if not pdf.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(pdf['manager_name'], pdf['total_career_points_given_away'])
    
    # Color the loser's bar red
    for i, bar in enumerate(bars):
        if pdf.iloc[i]['manager_name'] == loser:
            bar.set_color('red')
        else:
            bar.set_color('green')
    
    ax.set_xlabel('Manager', fontsize=12)
    ax.set_ylabel('Total Career Points Given Away', fontsize=12)
    ax.set_title(f'THE WORST TRADE OF ALL TIME\nSeason {worst_trade_season} - Differential: {magnitude} points', 
                 fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    display(fig)
else:
    print("No data to plot")

## Asset Breakdown Visualization

In [ ]:
import matplotlib.pyplot as plt

pdf_assets = worst_trade_breakdown.toPandas()

if not pdf_assets.empty:
    # Get unique managers
    managers = pdf_assets['manager_name'].unique()
    
    fig, axes = plt.subplots(1, len(managers), figsize=(15, 8), sharey=True)
    
    if len(managers) == 1:
        axes = [axes]
    
    for idx, manager in enumerate(managers):
        manager_data = pdf_assets[pdf_assets['manager_name'] == manager]
        manager_data = manager_data.sort_values('career_points_given_away', ascending=True)
        
        # Color code by asset type
        colors = ['#FF6B6B' if asset_type == 'player' else '#4ECDC4' 
                  for asset_type in manager_data['asset_type']]
        
        axes[idx].barh(manager_data['asset_name'], manager_data['career_points_given_away'], color=colors)
        axes[idx].set_xlabel('Career Points', fontsize=10)
        
        # Add red border if this is the loser
        if manager == loser:
            axes[idx].set_title(f'{manager}\n(LOSER)', fontsize=12, fontweight='bold', color='red')
            for spine in axes[idx].spines.values():
                spine.set_edgecolor('red')
                spine.set_linewidth(3)
        else:
            axes[idx].set_title(f'{manager}\n(Winner)', fontsize=12, fontweight='bold', color='green')
        
        axes[idx].tick_params(axis='y', labelsize=8)
    
    axes[0].set_ylabel('Asset', fontsize=10)
    plt.suptitle(f'Assets Given Away - THE WORST TRADE\nTransaction: {worst_trade_id}', 
                 fontsize=14, fontweight='bold')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#FF6B6B', label='Player'),
        Patch(facecolor='#4ECDC4', label='Draft Pick')
    ]
    fig.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    display(fig)
else:
    print("No asset data to plot")

---

## Analysis Complete

This notebook reveals:
1. The top 5 worst trades by career value differential
2. Complete breakdown of THE worst trade
3. What each side gave away and how many career points those assets scored
4. Visual representation of who got fleeced

**Methodology:** This calculates the full career value of assets given away, regardless of who owned them after the trade. It's the "hindsight is 20/20" worst trade analysis.